# 01 — PowerShell Corpus Assembly

**Project:** HierFuse — Hierarchical PowerShell / LotL Detection with Learned Fusion  
**Platform:** Kaggle Free Tier · CPU · Internet enabled  
**Runtime:** ~120–150 min  
**Outputs:** `corpus/powershell_corpus_v3.parquet`, `corpus/test_cross_source.parquet`, `corpus/corpus_stats_v3.json`

## Overview

Assembles the ~19,400-script PowerShell corpus from 30+ labelled sources:

| Class | Sources |
|---|---|
| **Malicious** | das-lab/mpsd, Fa2y, dessertlab/offensive-powershell (HF), GhostPack/Invoke-Evasion, Empire/PoshC2/Nishang/PowerSploit, Atomic Red Team, Caldera/Stockpile/EMU, SigmaHQ, BloodHound, WMIOps, theZoo, MalwareBazaar |
| **Benign** | GitHub (13 topic domains), DSC Community, dbatools, Microsoft (PSScriptAnalyzer, Azure-PS), Pester, PS Gallery, NuGet/Chocolatey, Ansible Windows, Azure Samples |

## Important design decisions

- **`obfusc_*` sources** (programmatic transforms of base scripts) are **only** collected here but  
  **quarantined from the training pool** in `02_splits.ipynb` → saved as `test_adversarial.parquet`.
- **50:50 balance** enforced by undersampling benign majority only (no SMOTE, per Arp et al. pitfall P3).
- **Cross-source holdout** carved out before balancing; sources disjoint from train/val/test.

## ATT&CK coverage target
`T1059.001, T1047, T1218, T1027, T1027.010, T1562.001, T1055, T1547.001, T1053.005,`  
`T1003.001, T1552, T1082, T1033, T1016, T1021.006, T1110, T1560, T1105, T1071.001, T1036, T1134, T1548`


## Cell 0 — Install dependencies and load secrets

In [1]:
import subprocess, sys, os, json, time, random, hashlib, re
import zipfile, io, shutil, csv
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
from typing import List, Dict, Optional

def pip(*pkgs):
    subprocess.run([sys.executable,'-m','pip','install','-q']+list(pkgs), check=True)

pip('requests','tqdm','pandas','pyarrow','PyYAML','py7zr','datasets','huggingface_hub')

import pandas as pd, requests, yaml
from tqdm.auto import tqdm

# ── Directories ───────────────────────────────────────────────────────────────
WORK = Path('/kaggle/working')
RAW  = WORK / 'raw'
PROC = WORK / 'corpus'
for d in [RAW, PROC]: d.mkdir(parents=True, exist_ok=True)

# ── Secrets (set via Kaggle Add-ons → Secrets) ────────────────────────────────
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN','')
HF_TOKEN     = os.environ.get('HF_TOKEN','')
MB_API_KEY   = os.environ.get('MB_API_KEY','')

try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    for key, var in [('GITHUB_TOKEN','GITHUB_TOKEN'),('HF_TOKEN','HF_TOKEN'),('MB_API_KEY','MB_API_KEY')]:
        try:
            val = _s.get_secret(key)
            globals()[var] = val
            if var == 'HF_TOKEN': os.environ['HF_TOKEN'] = val
            print(f'{key}: loaded')
        except Exception: print(f'{key}: not in secrets (using env or empty)')
except Exception: print('Kaggle secrets unavailable — using environment variables')

OUTPUT_PARQUET  = PROC / 'powershell_corpus_v3.parquet'
HOLDOUT_PARQUET = PROC / 'test_cross_source.parquet'
STATS_PATH      = PROC / 'corpus_stats_v3.json'

random.seed(42)
print(f'Working dir : {WORK}')
print(f'GitHub token: {"present" if GITHUB_TOKEN else "MISSING — rate limited to 60 req/hr"}')
print(f'HF token    : {"present" if HF_TOKEN else "missing (public datasets only)"}')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.3/71.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.3/495.3 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.6/100.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 8.0 MB/s eta 0:00:00
GITHUB_TOKEN: loaded
HF_TOKEN: loaded
MB_API_KEY: loaded
Working dir : /kaggle/working
GitHub token: present
HF token    : present


## Cell 1 — Schema helpers, dedup, git clone, GitHub API

In [2]:
import logging
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s | %(levelname)s | %(message)s',
                    datefmt='%H:%M:%S')
log = logging.getLogger('corpus')

# ── Global dedup via SHA-256 of normalised text ───────────────────────────────
SEEN: set = set()
ALL:  List[Dict] = []

def script_sha256(text: str) -> str:
    n = re.sub(r'\s+', ' ', text.lower().strip())
    return hashlib.sha256(n.encode('utf-8', errors='replace')).hexdigest()

def make_record(text, source, label, attck_ids=None, first_seen=None):
    text = str(text).strip()
    toks = len(text.split())
    if not (5 <= toks <= 80_000): return None
    sha = script_sha256(text)
    if sha in SEEN: return None
    SEEN.add(sha)
    return {'sha256': sha, 'source': source, 'label': int(label),
            'script_text': text, 'first_seen_date': first_seen,
            'token_count': toks, 'attck_ids': json.dumps(sorted(set(attck_ids or [])))}

def add(recs): ALL.extend(r for r in recs if r)

def git_clone(url, target, timeout=180, depth=1):
    if target.exists() and any(target.iterdir()):
        log.info(f'  Already present: {target.name}'); return True
    target.mkdir(parents=True, exist_ok=True)
    r = subprocess.run(['git','clone',f'--depth',str(depth),'--quiet',url,str(target)],
                       capture_output=True, timeout=timeout)
    if r.returncode != 0:
        log.warning(f'  Clone FAILED: {url[:70]}  {r.stderr.decode()[:100]}')
        shutil.rmtree(target, ignore_errors=True); return False
    log.info(f'  Cloned (depth={depth}): {target.name}'); return True

def extract_from_dir(base, source_tag, label, max_n=5000):
    if not base.exists(): return []
    files = list(base.rglob('*.ps1')) + list(base.rglob('*.psm1'))
    files = [f for f in files if not any(p in str(f) for p in ['.git','__pycache__'])]
    if len(files) > max_n: files = random.sample(files, max_n)
    recs = []
    for f in files:
        try:
            text = f.read_text(encoding='utf-8', errors='replace')
            rec = make_record(text, source_tag, label)
            if rec: recs.append(rec)
        except Exception: pass
    return recs

GH_HDRS = {'Accept':'application/vnd.github.v3+json','User-Agent':'lotl-fusion-research/3.0'}
if GITHUB_TOKEN: GH_HDRS['Authorization'] = f'token {GITHUB_TOKEN}'

def gh_get_repos(topic, max_repos=30):
    repos, page = [], 1
    while len(repos) < max_repos:
        try:
            r = requests.get('https://api.github.com/search/repositories',
                params={'q':f'topic:{topic} language:PowerShell','sort':'stars',
                        'order':'desc','per_page':30,'page':page},
                headers=GH_HDRS, timeout=30)
            if r.status_code == 403:
                wait = int(r.headers.get('Retry-After',60))
                log.warning(f'  GH rate limit — sleeping {wait}s'); time.sleep(wait); continue
            if r.status_code != 200: break
            items = r.json().get('items',[]); 
            if not items: break
            repos.extend(items)
            if len(items) < 30: break
            page += 1; time.sleep(1.5)
        except Exception as e: log.warning(f'  gh_get_repos: {e}'); break
    return repos[:max_repos]

def clone_repos_and_extract(repos, source_tag, label, max_files_total=1000, depth=1):
    recs, total = [], 0
    for repo in repos:
        if total >= max_files_total: break
        clone_url = repo.get('clone_url','')
        if not clone_url: continue
        target = RAW / ('gh_' + repo.get('full_name','').replace('/','_')[:40])
        if not git_clone(clone_url, target, depth=depth): continue
        r = extract_from_dir(target, source_tag, label, max_n=min(200, max_files_total-total))
        recs.extend(r); total += len(r)
        log.info(f'    {repo.get("full_name","?")}: {len(r):,} (total {total:,})')
    return recs

def odata_get_packages(api_url, max_pkgs=400):
    NS = {'atom':'http://www.w3.org/2005/Atom','d':'http://schemas.microsoft.com/ado/2007/08/dataservices'}
    pkgs, skip = [], 0
    while len(pkgs) < max_pkgs:
        try:
            r = requests.get(api_url, headers={'User-Agent':'lotl/3.0','Accept':'application/atom+xml'},
                             timeout=30, params={'$filter':'IsLatestVersion',
                             '$orderby':'DownloadCount desc','$top':100,'$skip':skip})
            if r.status_code != 200: break
            root = ET.fromstring(r.text)
            entries = root.findall('atom:entry', NS)
            if not entries: break
            for e in entries:
                dl = e.find('.//d:PackageDownloadUrl', NS)
                if dl is not None and dl.text: pkgs.append(dl.text)
            skip += 100; time.sleep(0.5)
        except Exception: break
    return pkgs[:max_pkgs]

print('Helpers ready.')


Helpers ready.


## Cell 2 — Malicious sources

In [3]:
# ── 2a: das-lab/mpsd ─────────────────────────────────────────────────────────
MPSD_DIR   = RAW / 'mpsd'
MPSD_ATTCK = ['T1059.001','T1027','T1082','T1033','T1016']
git_clone('https://github.com/das-lab/mpsd.git', MPSD_DIR, depth=3)
mpsd_recs = []
if MPSD_DIR.exists():
    for ps1 in MPSD_DIR.rglob('*.ps1'):
        parts = str(ps1).lower()
        lbl = 1 if any(x in parts for x in ['malware','malicious','attack']) else (
              0 if any(x in parts for x in ['benign','clean','legit']) else -1)
        if lbl == -1: continue
        try:
            text = ps1.read_text(encoding='utf-8', errors='replace')
            rec = make_record(text, f'mpsd_{"malicious" if lbl else "benign"}', lbl,
                              MPSD_ATTCK if lbl else None)
            if rec: mpsd_recs.append(rec)
        except Exception: pass
log.info(f'mpsd: {len(mpsd_recs):,}'); add(mpsd_recs)

# ── 2b: Fa2y/Malicious-PowerShell-Dataset ────────────────────────────────────
FA2Y_DIR = RAW / 'fa2y'
git_clone('https://github.com/Fa2y/Malicious-PowerShell-Dataset.git', FA2Y_DIR)
fa2y_recs = extract_from_dir(FA2Y_DIR, 'fa2y_malicious', 1)
for r in fa2y_recs: r['attck_ids'] = json.dumps(['T1059.001','T1047','T1218'])
log.info(f'Fa2y: {len(fa2y_recs):,}'); add(fa2y_recs)

# ── 2c: dessertlab/offensive-powershell (HuggingFace) ────────────────────────
dessert_recs = []
try:
    from datasets import load_dataset
    ds_dict = load_dataset('dessertlab/offensive-powershell', token=HF_TOKEN or None)
    DS_ATTCK = ['T1059.001','T1071.001','T1560','T1105']
    for split_name, ds in ds_dict.items():
        for row in tqdm(ds, desc=f'dessertlab/{split_name}', leave=False):
            text_val = None
            for col in ['command','cmd','script','text','content','output','input']:
                if col in row and isinstance(row[col], str) and row[col].strip():
                    text_val = row[col]; break
            if not text_val: continue
            rec = make_record(text_val, 'dessertlab_offensive', 1, DS_ATTCK)
            if rec: dessert_recs.append(rec)
except Exception as e: log.warning(f'dessertlab failed: {e}')
log.info(f'dessertlab: {len(dessert_recs):,}'); add(dessert_recs)

# ── 2d: GhostPack/Invoke-Evasion (.7z archives) ──────────────────────────────
import py7zr
EVASION_DIR  = RAW / 'invoke_evasion'
EVASION_ATTCK = ['T1027','T1027.010']
git_clone('https://github.com/GhostPack/Invoke-Evasion.git', EVASION_DIR, depth=3)
evasion_recs = []
for url in ['https://github.com/GhostPack/Invoke-Evasion/raw/master/Invoke-Evasion/Invoke-Evasion.7z']:
    try:
        r = requests.get(url, headers={'User-Agent':'Mozilla/5.0'}, timeout=60)
        if r.status_code != 200: continue
        with py7zr.SevenZipFile(io.BytesIO(r.content), mode='r') as arch:
            for fname, bio in arch.read().items():
                if not (fname.endswith('.ps1') or fname.endswith('.psm1')): continue
                text = bio.read().decode('utf-8', errors='replace')
                rec = make_record(text, 'invoke_evasion', 1, EVASION_ATTCK)
                if rec: evasion_recs.append(rec)
    except Exception as e: log.warning(f'Invoke-Evasion archive: {e}')
if not evasion_recs:
    evasion_recs = extract_from_dir(EVASION_DIR, 'invoke_evasion', 1, max_n=200)
    for r in evasion_recs: r['attck_ids'] = json.dumps(EVASION_ATTCK)
log.info(f'Invoke-Evasion: {len(evasion_recs):,}'); add(evasion_recs)

# ── 2e: Red-team frameworks ───────────────────────────────────────────────────
REDTEAM_REPOS = {
    'empire':      ('https://github.com/BC-SECURITY/Empire.git',       ['T1059.001','T1021.006','T1071.001','T1003.001','T1547.001','T1053.005'], 3),
    'poshc2':      ('https://github.com/nettitude/PoshC2.git',         ['T1059.001','T1021.006','T1071.001'], 3),
    'nishang':     ('https://github.com/samratashok/nishang.git',      ['T1059.001','T1082','T1033','T1016','T1003.001','T1047'], 3),
    'powersploit': ('https://github.com/PowerShellMafia/PowerSploit.git',['T1059.001','T1055','T1547.001','T1053.005','T1134'], 3),
}
redteam_recs = []
for name, (url, attck, depth) in REDTEAM_REPOS.items():
    target = RAW / f'redteam_{name}'
    git_clone(url, target, timeout=180, depth=depth)
    recs = extract_from_dir(target, f'redteam_{name}', 1, max_n=500)
    for r in recs: r['attck_ids'] = json.dumps(attck)
    log.info(f'  {name}: {len(recs):,}'); redteam_recs.extend(recs)
log.info(f'Red-team total: {len(redteam_recs):,}'); add(redteam_recs)

# ── 2f: Atomic Red Team (ATT&CK-mapped PS executors) ─────────────────────────
ATOMIC_DIR = RAW / 'atomic_red_team'
git_clone('https://github.com/redcanaryco/atomic-red-team.git', ATOMIC_DIR, timeout=300, depth=1)
atomic_recs = []
for yaml_file in tqdm(list((ATOMIC_DIR/'atomics').rglob('*.yaml')) +
                       list((ATOMIC_DIR/'atomics').rglob('*.yml')), desc='atomic', leave=False):
    try:
        with open(yaml_file, errors='replace') as fh: data = yaml.safe_load(fh)
        if not isinstance(data, dict): continue
        technique = data.get('attack_technique','')
        for test in data.get('atomic_tests', []):
            exec_block = test.get('executor', {})
            exec_name  = str(exec_block.get('name','') or '')
            is_ps = ('powershell' in exec_name.lower() or
                     'windows' in str(test.get('supported_platforms',[])).lower())
            if not is_ps: continue
            command = exec_block.get('command','') or exec_block.get('steps','')
            if not command or not isinstance(command, str): continue
            rec = make_record(command, 'atomic_red_team', 1, [technique] if technique else None)
            if rec: atomic_recs.append(rec)
    except Exception: continue
log.info(f'Atomic Red Team: {len(atomic_recs):,}'); add(atomic_recs)

# ── 2g: MITRE Caldera + Stockpile + EMU ──────────────────────────────────────
CALDERA_REPOS = {
    'caldera':   'https://github.com/mitre/caldera.git',
    'stockpile': 'https://github.com/mitre/stockpile.git',
    'emu':       'https://github.com/mitre/emu.git',
}
CALDERA_ATTCK = ['T1059.001','T1047','T1082','T1033']
caldera_recs  = []
for name, url in CALDERA_REPOS.items():
    target = RAW / f'caldera_{name}'
    git_clone(url, target, timeout=180)
    for yml in tqdm(list(target.rglob('*.yml'))+list(target.rglob('*.yaml')), desc=name, leave=False):
        try:
            with open(yml, errors='replace') as fh: data = yaml.safe_load(fh)
            if not isinstance(data, dict): continue
            for executor in data.get('executors', []):
                if not isinstance(executor, dict): continue
                ex_name = str(executor.get('name','') or '')
                ex_plat = str(executor.get('platform','') or '')
                if not ('psh' in ex_name or 'powershell' in ex_name or 'psh' in ex_plat): continue
                command = executor.get('command','') or executor.get('script','')
                if not command or not isinstance(command, str): continue
                rec = make_record(command, f'caldera_{name}', 1, CALDERA_ATTCK)
                if rec: caldera_recs.append(rec)
        except Exception: continue
log.info(f'Caldera/Stockpile/EMU: {len(caldera_recs):,}'); add(caldera_recs)

# ── 2h: SigmaHQ PS detection rules ───────────────────────────────────────────
SIGMA_DIR = RAW / 'sigma'
git_clone('https://github.com/SigmaHQ/sigma.git', SIGMA_DIR, timeout=300)
sigma_recs = []
for rule_dir in ['rules','rules-emerging-threats','rules-threat-hunting']:
    for yml in tqdm(list((SIGMA_DIR/rule_dir).rglob('*.yml'))
                    if (SIGMA_DIR/rule_dir).exists() else [], desc=rule_dir, leave=False):
        try:
            with open(yml, errors='replace') as fh: data = yaml.safe_load(fh)
            if not isinstance(data, dict): continue
            detect = data.get('detection', {})
            def flatten(obj):
                if isinstance(obj, str): return [obj]
                if isinstance(obj, list): return [x for item in obj for x in flatten(item)]
                if isinstance(obj, dict): return [x for v in obj.values() for x in flatten(v)]
                return []
            frags = [f for f in flatten(detect) if isinstance(f,str) and
                     any(kw in f.lower() for kw in ['powershell','invoke-','bypass','encoded',
                                                     'iex','webclient','downloadstring'])]
            if not frags: continue
            combined = '\n'.join(frags[:20])
            tags = [t for t in data.get('tags',[]) if t.startswith('attack.t')]
            rec = make_record(combined, 'sigma_rule', 1,
                              [t.upper().replace('attack.','') for t in tags[:3]])
            if rec: sigma_recs.append(rec)
        except Exception: continue
log.info(f'SigmaHQ: {len(sigma_recs):,}'); add(sigma_recs)

# ── 2i: BloodHound / WMIOps / Invoke-TheHash / PowerUpSQL ────────────────────
EXTRA_MAL = {
    'bloodhound_ps': ('https://github.com/BloodHoundAD/BloodHound.git',  ['T1059.001','T1082','T1033','T1016'], 200, 3),
    'wmiops':        ('https://github.com/ChrisTruncer/WMIOps.git',       ['T1059.001','T1047'],               100, 1),
    'invoke_thehash':('https://github.com/Kevin-Robertson/Invoke-TheHash.git', ['T1059.001','T1110'],          100, 1),
    'powerupsql':    ('https://github.com/NetSPI/PowerUpSQL.git',         ['T1059.001','T1552'],               200, 1),
}
extra_mal_recs = []
for name, (url, attck, max_f, depth) in EXTRA_MAL.items():
    target = RAW / f'extra_mal_{name}'
    git_clone(url, target, timeout=180, depth=depth)
    recs = extract_from_dir(target, f'extra_mal_{name}', 1, max_n=max_f)
    for r in recs: r['attck_ids'] = json.dumps(attck)
    log.info(f'  {name}: {len(recs):,}'); extra_mal_recs.extend(recs)
log.info(f'Extra malicious: {len(extra_mal_recs):,}'); add(extra_mal_recs)

n_mal = sum(1 for r in ALL if r['label']==1)
n_ben = sum(1 for r in ALL if r['label']==0)
log.info(f'=== MALICIOUS COLLECTION: {n_mal:,} (total so far: {len(ALL):,}) ===')


02:02:18 | INFO |   Cloned (depth=3): mpsd
02:02:25 | INFO | mpsd: 10,217
02:02:50 | INFO |   Cloned (depth=1): fa2y
02:03:25 | INFO | Fa2y: 1,809
02:03:26 | INFO | TensorFlow version 2.19.0 available.
02:03:26 | INFO | JAX version 0.7.2 available.
02:03:29 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/dessertlab/offensive-powershell/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
02:03:29 | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/dessertlab/offensive-powershell/1b71c152af5ebc8701c475c77097adcb5928b711/README.md "HTTP/1.1 200 OK"
02:03:29 | INFO | HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/dessertlab/offensive-powershell/1b71c152af5ebc8701c475c77097adcb5928b711/README.md "HTTP/1.1 200 OK"


README.md:   0%|          | 0.00/537 [00:00<?, ?B/s]

02:03:29 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/dessertlab/offensive-powershell/resolve/1b71c152af5ebc8701c475c77097adcb5928b711/offensive-powershell.py "HTTP/1.1 404 Not Found"
02:03:30 | INFO | HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/dessertlab/offensive-powershell/dessertlab/offensive-powershell.py "HTTP/1.1 404 Not Found"
02:03:30 | INFO | HTTP Request: GET https://huggingface.co/api/datasets/dessertlab/offensive-powershell/revision/1b71c152af5ebc8701c475c77097adcb5928b711 "HTTP/1.1 200 OK"
02:03:30 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/dessertlab/offensive-powershell/resolve/1b71c152af5ebc8701c475c77097adcb5928b711/.huggingface.yaml "HTTP/1.1 404 Not Found"
02:03:31 | INFO | HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=dessertlab/offensive-powershell "HTTP/1.1 200 OK"
02:03:31 | INFO | HTTP Request: GET https://huggingface.co/api/datasets/dessertlab/offensive-powershel

data/train-00000-of-00001-25809ad170aa95(…):   0%|          | 0.00/85.2k [00:00<?, ?B/s]

02:03:34 | INFO | HTTP Request: HEAD https://huggingface.co/datasets/dessertlab/offensive-powershell/resolve/1b71c152af5ebc8701c475c77097adcb5928b711/data/test-00000-of-00001-95b5b6f3a27c8cce.parquet "HTTP/1.1 302 Found"


...

02:05:35 | INFO | Caldera/Stockpile/EMU: 0
02:05:37 | INFO |   Cloned (depth=1): sigma


rules:   0%|          | 0/3132 [00:00<?, ?it/s]

rules-emerging-threats:   0%|          | 0/457 [00:00<?, ?it/s]

rules-threat-hunting:   0%|          | 0/139 [00:00<?, ?it/s]

02:05:48 | INFO | SigmaHQ: 84
02:05:55 | INFO |   Cloned (depth=3): extra_mal_bloodhound_ps
02:05:55 | INFO |   bloodhound_ps: 0
02:05:56 | INFO |   Cloned (depth=1): extra_mal_wmiops
02:05:56 | INFO |   wmiops: 1
02:05:57 | INFO |   Cloned (depth=1): extra_mal_invoke_thehash
02:05:57 | INFO |   invoke_thehash: 6
02:05:59 | INFO |   Cloned (depth=1): extra_mal_powerupsql
02:06:00 | INFO |   powerupsql: 12
02:06:00 | INFO | Extra malicious: 19
02:06:00 | INFO | === MALICIOUS COLLECTION: 8,944 (total so far: 13,248) ===


## Cell 3 — Benign sources

In [4]:
# ── 3a: GitHub benign repos (13 topic domains) ───────────────────────────────
BENIGN_DOMAINS = [
    ('sysadmin',              'github_sysadmin',   15, 600),
    ('active-directory',      'github_ad_admin',   12, 500),
    ('windows-administration','github_win_admin',  10, 400),
    ('azure-powershell',      'github_azure_ps',   12, 500),
    ('devops',                'github_devops',     10, 400),
    ('threat-hunting',        'github_threat_hunt',12, 500),
    ('blue-team',             'github_blue_team',  10, 400),
    ('dfir',                  'github_dfir',       10, 400),
    ('incident-response',     'github_ir',          8, 300),
    ('monitoring',            'github_monitoring', 10, 400),
    ('networking',            'github_networking',  8, 300),
    ('ci-cd',                 'github_cicd',        8, 300),
    ('group-policy',          'github_gpo',         6, 200),
]
gh_benign = []
for topic, tag, max_repos, max_files in BENIGN_DOMAINS:
    repos = gh_get_repos(topic, max_repos=max_repos)
    recs  = clone_repos_and_extract(repos, tag, 0, max_files_total=max_files)
    log.info(f'  {tag}: {len(recs):,}'); gh_benign.extend(recs); time.sleep(2)
log.info(f'GitHub benign total: {len(gh_benign):,}'); add(gh_benign)

# ── 3b: DSC Community ────────────────────────────────────────────────────────
DSC_REPOS = {
    'NetworkingDsc':         'https://github.com/dsccommunity/NetworkingDsc.git',
    'ComputerManagementDsc': 'https://github.com/dsccommunity/ComputerManagementDsc.git',
    'ActiveDirectoryDsc':    'https://github.com/dsccommunity/ActiveDirectoryDsc.git',
    'SecurityPolicyDsc':     'https://github.com/dsccommunity/SecurityPolicyDsc.git',
    'StorageDsc':            'https://github.com/dsccommunity/StorageDsc.git',
}
dsc_recs = []
for name, url in DSC_REPOS.items():
    target = RAW / f'dsc_{name}'
    git_clone(url, target); recs = extract_from_dir(target, f'dsc_{name}', 0, max_n=300)
    log.info(f'  {name}: {len(recs):,}'); dsc_recs.extend(recs)
log.info(f'DSC: {len(dsc_recs):,}'); add(dsc_recs)

# ── 3c: dbatools + Microsoft repos ───────────────────────────────────────────
DBA_DIR = RAW / 'dbatools'
git_clone('https://github.com/dataplat/dbatools.git', DBA_DIR, timeout=300, depth=3)
dba_recs = extract_from_dir(DBA_DIR, 'dbatools', 0, max_n=800)
log.info(f'dbatools: {len(dba_recs):,}'); add(dba_recs)

for name, url, max_f in [
    ('psscriptanalyzer', 'https://github.com/PowerShell/PSScriptAnalyzer.git', 400),
    ('azure_ps',         'https://github.com/Azure/azure-powershell.git', 400),
]:
    target = RAW / f'ms_{name}'
    git_clone(url, target, timeout=300, depth=1)
    recs = extract_from_dir(target, f'microsoft_{name}', 0, max_n=max_f)
    log.info(f'  microsoft_{name}: {len(recs):,}'); add(recs)

# ── 3d: Pester tests ─────────────────────────────────────────────────────────
pester_recs = []
for name, url in {'pester':'https://github.com/pester/Pester.git',
                  'vscode_ps':'https://github.com/PowerShell/vscode-powershell.git'}.items():
    target = RAW / f'pester_{name}'
    git_clone(url, target)
    recs = extract_from_dir(target, f'pester_{name}', 0, max_n=300)
    pester_recs.extend(recs)
log.info(f'Pester: {len(pester_recs):,}'); add(pester_recs)

# ── 3e: PowerShell Gallery ────────────────────────────────────────────────────
gallery_pkgs = odata_get_packages('https://www.powershellgallery.com/api/v2/Packages', max_pkgs=500)
gallery_recs = []
for pkg_url in tqdm(gallery_pkgs[:400], desc='gallery', leave=False):
    try:
        r = requests.get(pkg_url, timeout=30)
        if r.status_code != 200: continue
        with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
            for fname in zf.namelist():
                if not (fname.endswith('.ps1') or fname.endswith('.psm1')): continue
                text = zf.read(fname).decode('utf-8', errors='replace')
                rec = make_record(text, 'ps_gallery', 0)
                if rec: gallery_recs.append(rec)
        time.sleep(0.3)
    except Exception: continue
log.info(f'PS Gallery: {len(gallery_recs):,}'); add(gallery_recs)

n_ben_now = sum(1 for r in ALL if r['label']==0)
log.info(f'=== BENIGN COLLECTION: {n_ben_now:,} (total corpus: {len(ALL):,}) ===')


02:06:03 | INFO |   Cloned (depth=1): gh_Disassembler0_Win10-Initial-Setup-Script
02:06:03 | INFO |     Disassembler0/Win10-Initial-Setup-Script: 2 (total 2)
02:06:04 | INFO |   Cloned (depth=1): gh_gordonbay_Windows-On-Reins
02:06:04 | INFO |     gordonbay/Windows-On-Reins: 1 (total 3)
02:06:06 | INFO |   Cloned (depth=1): gh_nkasco_IT-Admin-Toolkit-WinUI
02:06:06 | INFO |     nkasco/IT-Admin-Toolkit-WinUI: 10 (total 13)
02:06:07 | INFO |   Cloned (depth=1): gh_GChuf_RCWM
02:06:07 | INFO |     GChuf/RCWM: 13 (total 26)
02:06:08 | INFO |   Cloned (depth=1): gh_thom-s_netsec-ps-scripts
02:06:08 | INFO |     thom-s/netsec-ps-scripts: 5 (total 31)
02:06:09 | INFO |   Cloned (depth=1): gh_Corsinvest_cv4pve-api-powershell
02:06:09 | INFO |     Corsinvest/cv4pve-api-powershell: 2 (total 33)
02:06:10 | INFO |   Cloned (depth=1): gh_ralish_dotfiles
02:06:10 | INFO |     ralish/dotfiles: 59 (total 92)
02:06:11 | INFO |   Cloned (depth=1): gh_Am0rphous_PowerShell
02:06:11 | INFO |     Am0rphous/

gallery: 0it [00:00, ?it/s]

02:20:29 | INFO | PS Gallery: 0
02:20:29 | INFO | === BENIGN COLLECTION: 9,830 (total corpus: 18,774) ===


## Cell 4 — Obfuscation variants (adversarial holdout — not in training pool)

In [5]:
import base64 as _b64, zlib

def obf_encode(s):
    enc = _b64.b64encode(s.encode('utf-16-le')).decode('ascii')
    return f'powershell -NoP -NonI -W Hidden -Exec Bypass -Enc {enc}'

def obf_compress(s):
    compressed = _b64.b64encode(zlib.compress(s.encode('utf-8', errors='replace'))).decode()
    return (f'$d=New-Object System.IO.MemoryStream;$b=[Convert]::FromBase64String("{compressed}");'
            f'$d.Write($b,0,$b.Length);$d.Position=0;'
            f'Invoke-Expression([System.IO.StreamReader]::new('
            f'New-Object System.IO.Compression.GZipStream($d,0))).ReadToEnd()')

def obf_string(s):
    parts = [f'[char]0x{ord(c):02x}' for c in s[:200]]
    return '(' + '+'.join(parts) + ')'

def obf_concat(s):
    words = s.split()[:50]
    return ';'.join(f'${chr(65+i)}="{w}"' for i,w in enumerate(words))

OBFUSC_FNS = {'obfusc_encode': obf_encode, 'obfusc_compress': obf_compress,
               'obfusc_string': obf_string, 'obfusc_concat':  obf_concat}

# Apply to first 500 unique malicious scripts
mal_base  = [r for r in ALL if r['label'] == 1][:500]
obfusc_recs = []
for tag, fn in OBFUSC_FNS.items():
    for base_rec in tqdm(mal_base, desc=tag, leave=False):
        try:
            text = fn(base_rec['script_text'])
            if not text or text == base_rec['script_text']: continue
            rec = make_record(text, tag, 1, json.loads(base_rec.get('attck_ids','[]')))
            if rec: obfusc_recs.append(rec)
        except Exception: pass

log.info(f'Obfusc variants (adversarial holdout only): {len(obfusc_recs):,}')
add(obfusc_recs)
log.info(f'Full corpus (pre-balance): {len(ALL):,}')


obfusc_encode:   0%|          | 0/500 [00:00<?, ?it/s]

obfusc_compress:   0%|          | 0/500 [00:00<?, ?it/s]

obfusc_string:   0%|          | 0/500 [00:00<?, ?it/s]

obfusc_concat:   0%|          | 0/500 [00:00<?, ?it/s]

02:20:31 | INFO | Obfusc variants (adversarial holdout only): 500
02:20:31 | INFO | Full corpus (pre-balance): 19,274


## Cell 5 — Cross-source holdout, balance, ATT&CK check, save

In [6]:
df_all = pd.DataFrame(ALL)
df_all['attck_ids'] = df_all['attck_ids'].apply(
    lambda x: json.loads(x) if isinstance(x,str) else (x or []))

# ── Cross-source holdout (carved before balance) ──────────────────────────────
# Sources with sufficient volume to spare 20% for holdout
HOLDOUT_SOURCES_BEN = {'ps_gallery', 'dbatools'}
HOLDOUT_SOURCES_MAL = {'atomic_red_team', 'sigma_rule'}

df_cross_parts = []
for src in HOLDOUT_SOURCES_BEN | HOLDOUT_SOURCES_MAL:
    sub = df_all[df_all['source'] == src]
    n_hold = max(50, int(len(sub)*0.20))
    df_cross_parts.append(sub.sample(n=min(n_hold, len(sub)), random_state=42))

df_cross = (pd.concat(df_cross_parts).drop_duplicates('sha256').reset_index(drop=True)
            if df_cross_parts else pd.DataFrame())
cross_shas = set(df_cross['sha256'])

# Remove cross-source + obfusc from training pool
OBFUSC_SOURCES = {'obfusc_encode','obfusc_compress','obfusc_string',
                  'obfusc_concat','obfusc_alias','obfusc_format','obfusc_token_HOLDOUT'}
df_obfusc = df_all[df_all['source'].isin(OBFUSC_SOURCES)].reset_index(drop=True)
df_pool   = df_all[~df_all['sha256'].isin(cross_shas) &
                   ~df_all['source'].isin(OBFUSC_SOURCES)].reset_index(drop=True)

n_b = (df_pool.label == 0).sum()
n_m = (df_pool.label == 1).sum()
target_n = min(n_b, n_m)
df_final = pd.concat([
    df_pool[df_pool.label==0].sample(n=target_n, random_state=42),
    df_pool[df_pool.label==1].sample(n=target_n, random_state=42),
]).sample(frac=1, random_state=42).reset_index(drop=True)

# ── ATT&CK coverage ──────────────────────────────────────────────────────────
REQUIRED_ATTCK = {
    'T1059.001','T1047','T1218','T1027','T1027.010','T1562.001','T1055',
    'T1547.001','T1053.005','T1003.001','T1552','T1082','T1033','T1016',
    'T1021.006','T1110','T1560','T1105','T1071.001','T1036','T1134','T1548',
}
covered = set()
for tags in df_final['attck_ids']:
    if isinstance(tags, list): covered.update(tags)
missing  = REQUIRED_ATTCK - covered
log.info(f'ATT&CK coverage: {len(covered & REQUIRED_ATTCK)}/{len(REQUIRED_ATTCK)}'
         f'{" — MISSING: "+str(missing) if missing else " — COMPLETE ✓"}')

# ── Save ─────────────────────────────────────────────────────────────────────
df_final['attck_ids'] = df_final['attck_ids'].apply(json.dumps)
df_final.to_parquet(OUTPUT_PARQUET, index=False, engine='pyarrow')
log.info(f'Corpus saved: {OUTPUT_PARQUET}  ({OUTPUT_PARQUET.stat().st_size/1e6:.1f} MB)')

if not df_cross.empty:
    df_cross['attck_ids'] = df_cross['attck_ids'].apply(
        lambda x: json.dumps(x) if isinstance(x,list) else (x or '[]'))
    df_cross.to_parquet(HOLDOUT_PARQUET, index=False, engine='pyarrow')
    log.info(f'Cross-source holdout saved: {HOLDOUT_PARQUET}')

df_obfusc['attck_ids'] = df_obfusc['attck_ids'].apply(
    lambda x: json.dumps(x) if isinstance(x,list) else (x or '[]'))
adv_path = PROC / 'test_adversarial.parquet'
df_obfusc.to_parquet(adv_path, index=False, engine='pyarrow')
log.info(f'Adversarial holdout saved: {adv_path}  ({len(df_obfusc):,} scripts)')

stats = {
    'version':     'v3',
    'n_total':     int(len(df_final)),
    'n_benign':    int((df_final.label==0).sum()),
    'n_malicious': int((df_final.label==1).sum()),
    'n_sources':   int(df_final.source.nunique()),
    'sources':     df_final.source.value_counts().to_dict(),
    'attck_covered': sorted(covered & REQUIRED_ATTCK),
    'attck_missing': sorted(missing),
    'n_cross_source': int(len(df_cross)),
    'n_adversarial':  int(len(df_obfusc)),
    'token_p50': float(df_final.token_count.quantile(0.50)),
    'token_p90': float(df_final.token_count.quantile(0.90)),
}
with open(STATS_PATH,'w') as f: json.dump(stats,f,indent=2)
log.info(f'Stats saved: {STATS_PATH}')

print('\n=== CORPUS ASSEMBLY COMPLETE ===')
print(f'Total (balanced): {stats["n_total"]:,}')
print(f'Benign/Malicious: {stats["n_benign"]:,} / {stats["n_malicious"]:,}')
print(f'Sources         : {stats["n_sources"]}')
print(f'ATT&CK coverage : {len(stats["attck_covered"])}/{len(REQUIRED_ATTCK)}')
print(f'Cross-source    : {stats["n_cross_source"]:,}')
print(f'Adversarial     : {stats["n_adversarial"]:,} (quarantined from training)')
print('\nNext: upload /kaggle/working/corpus/ as Kaggle Dataset "01-corpus-dataset"')
print('Then run 02_splits.ipynb')

import shutil
import os

# Clean up the raw directory to remove invalid file names 
# and drastically reduce the Kaggle dataset output size.
raw_dir = '/kaggle/working/raw'
if os.path.exists(raw_dir):
    shutil.rmtree(raw_dir)
    print(f"\nCleaned up {raw_dir}")

02:20:31 | INFO | ATT&CK coverage: 20/22 — MISSING: {'T1562.001', 'T1548'}
02:20:43 | INFO | Corpus saved: /kaggle/working/corpus/powershell_corpus_v3.parquet  (510.5 MB)
02:20:43 | INFO | Cross-source holdout saved: /kaggle/working/corpus/test_cross_source.parquet
02:20:43 | INFO | Adversarial holdout saved: /kaggle/working/corpus/test_adversarial.parquet  (500 scripts)
02:20:43 | INFO | Stats saved: /kaggle/working/corpus/corpus_stats_v3.json



=== CORPUS ASSEMBLY COMPLETE ===
Total (balanced): 17,396
Benign/Malicious: 8,698 / 8,698
Sources         : 36
ATT&CK coverage : 20/22
Cross-source    : 406
Adversarial     : 500 (quarantined from training)

Next: upload /kaggle/working/corpus/ as Kaggle Dataset "01-corpus-dataset"
Then run 02_splits.ipynb

Cleaned up /kaggle/working/raw
